In [1]:
import pandas as pd

In [2]:
import requests

In [3]:
import json

In [4]:
import mysql.connector

In [5]:
#COMPETITIONS DATA

In [6]:
comp_url = "https://api.sportradar.com/tennis/trial/v3/en/competitions.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [7]:
headers = {"accept": "application/json"}

In [8]:
response = requests.get(comp_url, headers=headers)

In [9]:
data=json.loads(response.text)

In [10]:
catdata=[]

In [11]:
compdata=[]

In [12]:
for i in data["competitions"]:
    catdata.append({
        "category_id":i["category"]["id"],
        "category_name":i["category"]["name"]
    })
    compdata.append({
          "comp_id":i["id"],
          "comp_name":i["name"],
          "parent_id":i.get("parent_id",None),
          "type":i["type"],
          "gender":i["gender"],
          "category_id":i["category"]["id"]
    })

In [13]:
df=pd.DataFrame(catdata)
df=df.where(pd.notnull(df),None)

In [18]:
df1=pd.DataFrame(compdata)
df1=df1.where(pd.notnull(df1),None)

In [19]:
#COMPLEXES DATA

In [20]:
complex_url="https://api.sportradar.com/tennis/trial/v3/en/complexes.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [21]:
headers = {"accept": "application/json"}

In [22]:
response2 = requests.get(complex_url, headers=headers)

In [23]:
data2=json.loads(response2.text)

In [24]:
complexdata=[]

In [25]:
venuedata=[]

In [26]:
for i in data2["complexes"]:
    complexdata.append({
        "complex_id":i["id"],
        "complex_name":i["name"]
    })
    for venue in i.get("venues",[]):
        venuedata.append({
            "venue_id":venue.get("id"),
            "venue_name":venue.get("name"),
            "city_name":venue.get("city_name"),
            "country_name":venue.get("country_name"),
            "country_code":venue.get("country_code"),
            "timezone":venue.get("timezone"),
            "complex_id":i.get("id")
        })

In [27]:
df2=pd.DataFrame(complexdata)

In [28]:
df3=pd.DataFrame(venuedata)

In [25]:
#RANKING DATA

In [27]:
ranking_url="https://api.sportradar.com/tennis/trial/v3/en/double_competitors_rankings.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [28]:
headers = {"accept": "application/json"}

In [29]:
response3 = requests.get(ranking_url, headers=headers)

In [30]:
data3=json.loads(response3.text)

In [31]:
competitor_rank=[]

In [61]:
competitor=[]

In [62]:
for i in data3.get("rankings",[]):
    for x in i.get("competitor_rankings",[]):
        y=x.get("competitor",[])
        competitor_rank.append({
         "rank_id":x.get("rank_id"),
         "rank":x.get("rank"),
         "movement":x.get("movement"),
         "points":x.get("points"),
         "competitions_played":x.get("competition_played"),
         "competitor_id":y.get("id")
        })
        competitor.append({
         "competitor_id":y.get("id"),
         "name":y.get("name"),
         "country":y.get("country"),
         "country_code":y.get("country_code"),
         "abbreviation":y.get("abbreviation")
        })

In [34]:
df4=pd.DataFrame(competitor_rank)

In [66]:
df5=pd.DataFrame(competitor)

In [42]:
#CONNECT SQLDATA

In [29]:
conn=mysql.connector.connect(host="localhost",user="root",password="Dhinesh@0512")

In [30]:
cursor=conn.cursor()

In [45]:
cursor.execute("CREATE DATABASE TENNIS_DATABASE")

In [31]:
cursor.execute("USE TENNIS_DATABASE")

In [ ]:
#CREATE SQL TABLE-CATEGORIES
cursor.execute("""
    CREATE TABLE CATEGORIES(
     `Category Id` VARCHAR(50),
     `Category Name` VARCHAR(100)
    );
""")

In [55]:
#INSERTING DATAS INTO CATEGORIES TABLE
q1 = """INSERT INTO CATEGORIES(`Category Id`, `Category Name`) VALUES (%s, %s)"""
values1 = df.values.tolist()
cursor.executemany(q1, values1)
conn.commit()

In [48]:
#CREATE SQL TABLE-COMPETITIONS
cursor.execute("""
    CREATE TABLE COMPETITIONS(
     `Competition Id` VARCHAR(50),
     `Competition Name` VARCHAR(100),
     `Parent Id` VARCHAR(50),
     `Type` VARCHAR(20),
     `Gender` VARCHAR(20),
     `Category Id` VARCHAR(50)
    );
""")

In [56]:
#INSERTING DATAS INTO COMPETITIONS TABLE
q2 = """INSERT INTO COMPETITIONS(`Competition Id`, `Competition Name`,`Parent Id`,`Type`,`Gender`,`Category Id`) VALUES (%s,%s,%s,%s,%s,%s)"""
values2 = df1.values.tolist()
cursor.executemany(q2, values2)
conn.commit()

In [50]:
#CREATE SQL TABLE-COMPLEXES
cursor.execute("""
    CREATE TABLE COMPLEXES(
     `Complex Id` VARCHAR(50),
     `Complex Name` VARCHAR(100)
    );
""")

In [32]:
#INSERTING DATAS INTO COMPLEXES TABLE
q3 = """INSERT INTO COMPLEXES(`Complex Id`, `Complex Name`) VALUES (%s, %s)"""
values3 = df2.values.tolist()
cursor.executemany(q3, values3)
conn.commit()

In [52]:
#CREATE SQL TABLE-VENUES
cursor.execute("""
    CREATE TABLE VENUES(
     `Venue Id` VARCHAR(50),
     `Venue Name` VARCHAR(100),
     `City Name` VARCHAR(100),
     `Country Name` VARCHAR(100),
     `Country Code` CHAR(3),
     `Timezone`VARCHAR(100),
     `Complex Id` VARCHAR(50)
    );
""")

In [59]:
#INSERTING DATAS INTO VENUES TABLE
q4 = """INSERT INTO VENUES(`Venue Id`,`Venue Name`,`City Name`,`Country Name`,`Country Code`,`Timezone`,`Complex Id`) VALUES (%s,%s,%s,%s,%s,%s,%s)"""
values4 = df3.values.tolist()
cursor.executemany(q4, values4)
conn.commit()

In [53]:
#CREATE SQL TABLE-COMPETITOR_RANKINGS
cursor.execute("""
    CREATE TABLE COMPETITOR_RANKINGS(
     `Rank Id` INT,
     `Rank` INT,
     `Movement` INT,
     `Points` INT,
     `Competitions Played` INT,
     `Competitor Id` VARCHAR(50)
    );
""")

In [60]:
#INSERTING DATAS INTO COMPETITOR_RANKINGS TABLE
q5 = """INSERT INTO COMPETITOR_RANKINGS(`Rank Id`,`Rank`,`Movement`,`Points`,`Competitions Played`,`Competitor Id`) VALUES (%s,%s,%s,%s,%s,%s)"""
values5 = df4.values.tolist()
cursor.executemany(q5, values5)
conn.commit()

In [54]:
#CREATE SQL TABLE- COMPETITORS
cursor.execute("""
    CREATE TABLE COMPETITORS(
     `Competitor Id` VARCHAR(50),
     `Name` VARCHAR(100),
     `Country` VARCHAR(100),
     `Country Code` CHAR(3),
     `Abbreviation` VARCHAR(10)
    );
""")

In [68]:
# INSERTING DATAS INTO COMPETITORS TABLE
q6 = """INSERT INTO COMPETITORS(`Competitor Id`,`Name`,`Country`,`Country Code`,`Abbreviation`) VALUES (%s,%s,%s,%s,%s)"""
values6 = df5.values.tolist()
cursor.executemany(q6, values6)
conn.commit()

In [53]:
#SQL QUERIES IN CATEGORIES AND COMPETITIONS TABLES

In [32]:
#LIST ALL COMPETITIONS ALONG WITH THEIR CATEGORY NAME
query1 = """
SELECT 
    c.`Competition Id` AS competition_id, 
    c.`Competition Name` AS competition_name, 
    cat.`Category Name`
FROM COMPETITIONS c
JOIN CATEGORIES cat ON c.`Category Id` = cat.`Category Id`
"""
cursor.execute(query1)
result1 = cursor.fetchall()
querydb1 = pd.DataFrame(result1, columns=["Competition id", "Competition name", "Category name"])

In [34]:
#COUNT THE NUMBER OF COMPETITIONS IN EACH CATEGORY
query2 = """
SELECT 
    cat.`Category Name` AS category_name,
    COUNT(c.`Competition Id`) AS competition_count
FROM COMPETITIONS c
JOIN CATEGORIES cat ON c.`Category Id` = cat.`Category Id`
GROUP BY cat.`Category Name`
ORDER BY competition_count DESC;
"""
cursor.execute(query2)
result2 = cursor.fetchall()
querydb2 = pd.DataFrame(result2, columns=["Category name", "Competition count"])

In [36]:
#FIND ALL COMPETITIONS OF TYPE 'DOUBLES'
query3 = """
SELECT 
    c.`Competition Id`,
    c.`Competition Name`
FROM COMPETITIONS c
WHERE c.`Type` = 'doubles';
"""
cursor.execute(query3)
result3 = cursor.fetchall()
querydb3 = pd.DataFrame(result3, columns=["Competition Id", "Competition Name"])

In [42]:
#GET COMPETITIONS THAT BELONG TO A SPECIFIC CATEGORY
category_name = input("Enter the Category Name")

query4 = """
SELECT 
    c.`Competition Id`,
    c.`Competition Name`,
    cat.`Category Name`
FROM COMPETITIONS c
JOIN CATEGORIES cat ON c.`Category Id` = cat.`Category Id`
WHERE cat.`Category Name` = %s;
"""
cursor.execute(query4, (category_name,))
result4 = cursor.fetchall()
querydb4 = pd.DataFrame(result4, columns=["Competition id", "Competition name","Category Name"])

Enter the Category Name ITF Men


In [44]:
query5 = """
SELECT 
    parent.`Competition Id` AS Parent_Competition_Id,
    parent.`Competition Name` AS Parent_Competition_Name,
    sub.`Competition Id` AS Sub_Competition_Id,
    sub.`Competition Name` AS Sub_Competition_Name
FROM COMPETITIONS parent
JOIN COMPETITIONS sub
    ON parent.`Competition Id` = sub.`Parent Id`
ORDER BY parent.`Competition Name`, sub.`Competition Name`;
"""
cursor.execute(query5)
result5 = cursor.fetchall()
querydb5 = pd.DataFrame(result5, columns=["Parent Competition Id", "Parent Competition Name", "Sub Competition Id", "Sub Competition Name"])


In [49]:
#ANALYZE THE DISTRIBUTION OF COMPETITION TYPES BY CATEGORY
query6 = """
SELECT 
    cat.`Category Name`,
    comp.`Type`,
    COUNT(*) AS competition_count
FROM COMPETITIONS comp
JOIN CATEGORIES cat
    ON comp.`Category Id` = cat.`Category Id`
GROUP BY cat.`Category Name`, comp.`Type`
ORDER BY cat.`Category Name`, comp.`Type`;
"""
cursor.execute(query6)
result6 = cursor.fetchall()
querydb6 = pd.DataFrame(result6, columns=["Category Name","Competition Type","Competition Count"])

In [50]:
#LIST ALL COMPETITIONS WITH NO PARENT
query7 = """
SELECT 
    `Competition Id`, 
    `Competition Name`
FROM COMPETITIONS
WHERE `Parent Id` IS NULL
ORDER BY `Competition Name`;
"""
cursor.execute(query7)
result7 = cursor.fetchall()
querydb7 = pd.DataFrame(result7, columns=["Competition Id", "Competition Name"])

In [54]:
#SQL QUERIES IN COMPLEXES AND VENUES TABLES

In [55]:
#LIST ALL VENUES ALONG WITH THEIR ASSOCIATED COMPLEX NAME
query8 = """
SELECT v.`Venue Name`, c.`Complex Name`
FROM VENUES v
JOIN COMPLEXES c ON v.`Complex Id` = c.`Complex Id`;
"""
cursor.execute(query8)
result8 = cursor.fetchall()
querydf8 = pd.DataFrame(result8, columns=["Venue Name","Complex Name"])

In [57]:
#COUNT THE NUMBER OF VENUES IN EACH COMPLEX
query9 = """
SELECT c.`Complex Name`, COUNT(v.`Venue Id`) AS `Venue Count`
FROM COMPLEXES c
LEFT JOIN VENUES v ON c.`Complex Id` = v.`Complex Id`
GROUP BY c.`Complex Id`, c.`Complex Name`;
"""
cursor.execute(query9)
result9 = cursor.fetchall()
querydf9 = pd.DataFrame(result9, columns=["Complex Name","Venue Count"])

In [60]:
#GET THE DETAILS OF VENUES IN A SPECIFIC COUNTRY
country_name=input("Enter the Country name")
query10 = """
SELECT v.`Venue Name`, v.`Country Name` 
FROM VENUES v
WHERE `Country Name`= %s;
"""
cursor.execute(query10,(country_name,))
result10 = cursor.fetchall()
querydf10 = pd.DataFrame(result10, columns=["Venue Name","Country Name"])

Enter the Country name Chile


In [66]:
#IDENTIFY ALL VENUES AND THEIR TIMEZONES
query11 = """
SELECT v.`Venue Name`, v.`Timezone` 
FROM VENUES v;
"""
cursor.execute(query11)
result11 = cursor.fetchall()
querydf11 = pd.DataFrame(result11, columns=["Venue Name","Timezone"])

In [64]:
#FIND COMPLEXES THAT HAVE MORE THAN ONE VENUE
query12 = """
SELECT c.`Complex Name`, COUNT(v.`Venue Id`) AS `Venue Count`
FROM COMPLEXES c
JOIN VENUES v ON c.`Complex Id` = v.`Complex Id`
GROUP BY c.`Complex Id`, c.`Complex Name`
HAVING COUNT(v.`Venue Id`) > 1;
"""
cursor.execute(query12)
result12 = cursor.fetchall()
querydf12 = pd.DataFrame(result12, columns=["Complex Name","Venue Count"])

In [67]:
#LIST VENUES GROUPED BY COUNTRY
query13 = """
SELECT v.`Country Name`, v.`Venue Name`
FROM VENUES v
ORDER BY v.`Country Name`, v.`Venue Name`;
"""
cursor.execute(query13)
result13 = cursor.fetchall()
querydf13 = pd.DataFrame(result13, columns=["Country Name","Venue Name"])
querydf13 = querydf13.groupby("Country Name")["Venue Name"].apply(list).reset_index()

In [69]:
#FIND ALL VENUES FOR A SPECIFIC COMPLEX
complex_name=input("Enter the Complex Name")
query14 = """
SELECT v.`Venue Name`, v.`City Name`,v.`Country Name`
FROM VENUES v
JOIN COMPLEXES c ON v.`Complex Id` = c.`Complex Id`
WHERE c.`Complex Name`= %s;
"""
cursor.execute(query14,(complex_name,))
result14 = cursor.fetchall()
querydf14 = pd.DataFrame(result14, columns=["Venue Name","City Name","Country Name"])

Enter the Complex Name Nacional


In [71]:
#SQL QUERIES IN COMPETITOR_RANKINGS AND COMPETITOR TABLE

In [78]:
#GET ALL COMPETITORS WITH THEIR RANK AND POINTS
query15 = """
SELECT c.`Name`, c.`Country`, r.`Rank`, r.`Points`
FROM COMPETITORS c
JOIN COMPETITOR_RANKINGS r ON c.`Competitor Id` = r.`Competitor Id`
ORDER BY r.`Rank` ASC;
"""
cursor.execute(query15)
result15 = cursor.fetchall()
querydf15 = pd.DataFrame(result15, columns=["Name","Country","Rank","Points"])

In [79]:
#FIND COMPETITORS RANKED IN THE TOP 5
query16 = """
SELECT c.`Name`, c.`Country`, r.`Rank`, r.`Points`
FROM COMPETITORS c
JOIN COMPETITOR_RANKINGS r ON c.`Competitor Id` = r.`Competitor Id`
WHERE r.`Rank`<=5
ORDER BY r.`Rank` ASC;
"""
cursor.execute(query16)
result16 = cursor.fetchall()
querydf16 = pd.DataFrame(result16, columns=["Name","Country","Rank","Points"])

In [80]:
#LIST COMPETITORS WITH NO RANK MOVEMENT
query17 = """
SELECT c.`Name`, c.`Country`, r.`Rank`, r.`Points`,r.`Movement`
FROM COMPETITORS c
JOIN COMPETITOR_RANKINGS r ON c.`Competitor Id` = r.`Competitor Id`
WHERE r.`Movement`=0
ORDER BY r.`Rank`;
"""
cursor.execute(query17)
result17 = cursor.fetchall()
querydf17 = pd.DataFrame(result17, columns=["Name","Country","Rank","Points","Movements"])

In [81]:
#GET THE TOTAL POINTS OF COMPETITORS FROM A SPECIFIC COUNTRY
country_name = input("Enter country name: ")
query18 = """
SELECT c.`Country`, SUM(r.`Points`) AS `Total Points`
FROM COMPETITORS c
JOIN COMPETITOR_RANKINGS r ON c.`Competitor Id` = r.`Competitor Id`
WHERE c.`Country` = %s
GROUP BY c.`Country`;
"""
cursor.execute(query18,(country_name,))
result18 = cursor.fetchall()
querydf18 = pd.DataFrame(result18, columns=["Country","Total Points"])

Enter country name:  Croatia


In [83]:
#COUNT THE NUMBER OF COMPETITORS PER COUNTRY
query19 = """
SELECT c.`Country`, COUNT(*) AS `Competitor Count`
FROM COMPETITORS c
GROUP BY c.`Country`
ORDER BY `Competitor Count` DESC;
"""
cursor.execute(query19)
result19 = cursor.fetchall()
querydf19 = pd.DataFrame(result19, columns=["Country","Competitor Count"])

In [84]:
#FIND COMPETITORS WITH THE HIGHEST POINTS
query20 = """
SELECT c.`Name`, c.`Country`, r.`Points`
FROM COMPETITORS c
JOIN COMPETITOR_RANKINGS r ON c.`Competitor Id` = r.`Competitor Id`
WHERE r.`Points` = (SELECT MAX(`Points`) FROM COMPETITOR_RANKINGS);"""
cursor.execute(query20)
result20 = cursor.fetchall()
querydf20 = pd.DataFrame(result20, columns=["Name","Country","Points"])

In [87]:
cursor.close()
conn.close()